In [10]:
from datetime import datetime
from IPython.display import display, Markdown
from zoneinfo import ZoneInfo

current_date = datetime.now(ZoneInfo('America/Mexico_City')).strftime("%Y-%m-%d %H:%M:%S")
formatted_text = f"""
# <font color='orange'> Fourier Analysis
<font color='green'> Made by: J. Daniel Arzate | Date: 2020 - Enero 19 </font>

<font color='red'> Las view: {current_date}</font>
"""
display(Markdown(formatted_text))


# <font color='orange'> Fourier Analysis
<font color='green'> Made by: J. Daniel Arzate | Date: 2020 - Enero 19 </font>

<font color='red'> Las view: 2026-11-11 11:45:08</font>


# Síntesis Estocástica y Determinista Ruido Blanco

En esta practica utilizaremos el **una sintetizador de osciladores ($N$-Tori)** para explorar la relación fundamental entre el **dominio del tiempo**, el **dominio de la frecuencia (PSD)** y el **dominio de la autocorrelación ($R_{xx}(\tau)$)**.

La forma canónica del sintetizador $N$-Tori en tiempo continuo es:

$$h(t) = \sum_{k=1}^{K} A_k \cos(2\pi f_k t + \phi_k)$$

donde $f_k = k \cdot \Delta f$ representa las frecuencias armónicas en la rejilla discreta $\Delta f = 1/T_{\text{total}}$.

A partir de este único modelo matemático, sintetizaremos tres regímenes físicos muy distintos modificando **únicamente** la estructura de sus amplitudes $A_k$ y sus fases $\phi_k$:

1. **Pulso Delta de Dirac:** Amplitudes constantes, frecuencias armónicas y **fases perfectamente coherentes ($\phi_k = 0$)**.
2. **Ruido Blanco Estocástico:** Amplitudes constantes, frecuencias armónicas y **fases aleatorias ($\phi_k \sim U[0, 2\pi)$)**.
3. **Ruido Rosa y Browniano (Ruido de Colores):** Amplitudes dependientes de la frecuencia ($A_k \propto 1/f^\alpha$) y **fases aleatorias ($\phi_k \sim U[0, 2\pi)$)**.


Nota: Recuerda construir los $f_k$ del sintetizador mediante $f_k = k \cdot \Delta f$ para evitar "leakage". Haz una función llamada ntori. Pista de los parámetros de mi función:


    def arz_Nftori(srate=None, sec=60, Nfrec=1000, beta=2, f_min=1, f_max=None, Ndata=1, plot=False,save=None, *args):
    
    Genera señales N-Tori con espectro 1/f^β
    Parámetros:
    -----------
    srate : float
        Frecuencia de muestreo
    sec : float
        Duración en segundos (default=60)
    Nfrec : int
        Número de señales aleatorias a sumar (default=2000)
    beta : float
        Exponente espectral (0=blanco, 1=rosa, 2=rojo) (default=2)
    f_min : float
        Frecuencia mínima (default=1)
    f_max : float
        Frecuencia máxima (default=srate/2)
    Ndata : int
        Número de series temporales a generar (default=1)
    op : bool
        Si True, muestra gráficos (default=False)
    
    Retorna:
    --------
    y : ndarray
        Señal(es) generada(s)
    op : bool
            Si True, muestra gráficos (default=False)    
---

## Qué es el Ruido
El definir Ruido es una tarea bastante ambigua ya que nos tenemos que centrar en un ambito en especifico para intentar desde este darle un significado.
Dentro de procesos estocasticos el ruido no es unicamente el `error de medición` va más allá siendo una VA subyacente $\{X(t)\}$ que posee sus propias propiedades estadísticas, distribuciones de probabilidad y estructura topológica en el dominio de las frecuencias.

Siendo que lo que más se adapta a los procesos estocasticos es el ruido blanco y café

In [11]:
# Función para N-tori 
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def base(srate=1000, sec=60, Nfrec=1000, beta=0, f_min=1, f_max=None, Ndata=1, plot=False, save=None, is_dirac=False):
    """
    Genera señales N-Tori con espectro 1/f^beta.
    """
    if srate is None:
        srate = 1000
    if f_max is None: 
        f_max = srate / 2
        
    t = np.arange(0, sec, 1/srate)
    df = 1 / sec # Resolución en frecuencia
    
    # Construcción de la rejilla de frecuencias para evitar "leakage"
    k_min = max(1, int(f_min / df))
    k_max = min(Nfrec, int(f_max / df))
    k = np.arange(k_min, k_max + 1)
    f_k = k * df
    
    # Amplitud dependiente de la frecuencia. 
    # El espectro de potencia S(f) decae como 1/f^beta, por lo que la amplitud decae como 1/f^(beta/2)
    A_k = 1.0 / (f_k ** (beta / 2))
    
    y = np.zeros((Ndata, len(t)))
    
    for i in range(Ndata):
        if is_dirac:
            phi_k = np.zeros_like(f_k) # Fases perfectamente coherentes (Delta de Dirac)
        else:
            phi_k = np.random.uniform(0, 2*np.pi, len(f_k)) # Fases estocásticas (Ruido)
            
        # Síntesis matemática del N-Tori
        signal = np.zeros_like(t)
        for A, f, phi in zip(A_k, f_k, phi_k):
            signal += A * np.cos(2 * np.pi * f * t + phi)
        
        # Normalización para visualizar ambas señales en la misma escala
        y[i, :] = signal / np.max(np.abs(signal))
        
    if plot:
        plt.figure(figsize=(10, 2))
        plt.plot(t, y[0], color='black')
        plt.title(f"Señal N-Tori ($\\beta$={beta}, Dirac={is_dirac})")
        plt.xlabel("Tiempo (s)")
        plt.show()
        
    return t, y[0]

In [ ]:
# 1. Generación de las señales (usamos 2 segundos para visualizar claramente)
srate = 1000
sec = 2
t, h_delta = base(srate=srate, sec=sec, beta=0, is_dirac=True)   # Pulso Delta
t, h_white = base(srate=srate, sec=sec, beta=0, is_dirac=False)  # Ruido Blanco Estocástico


# --- B) Autocorrelación R_xx ---
tau = np.arange(-len(h_delta)+1, len(h_delta)) / srate
Rxx_delta = np.correlate(h_delta, h_delta, mode='full')
Rxx_white = np.correlate(h_white, h_white, mode='full')

# --- C) Espectro de Potencias (PSD) ---
f_fft = np.fft.rfftfreq(len(t), d=1/srate)
psd_delta = np.abs(np.fft.rfft(h_delta))**2
psd_white = np.abs(np.fft.rfft(h_white))**2

# --- D) Espectro de Fases ---
fases_delta = np.angle(np.fft.rfft(h_delta))
fases_white = np.angle(np.fft.rfft(h_white))

# --- E) Plano Complejo ---
H_delta = np.fft.rfft(h_delta)[1:200]
H_white = np.fft.rfft(h_white)[1:200]

## ⭕ Quest: Destruyendo la Coherencia de Fase

### Delta de Dirac
* Grafica el comportamiento en el dominio del tiempo $h(t)$ del **Pulso Delta  Dirac y Ruido Blanco Estocástico**. A pesar de estar constituido por cientos de oscilaciones continuas, las ondas se cancelan destructivamente en casi todo el tiempo excepto en $t = 0$. ¿Qué propiedad de la alineación de fase ($\phi_k = 0$) genera esta interferencia instantánea? <br>
$R:$ interferencia de ondas


In [27]:
fig_tiempo = make_subplots(rows=1, cols=2, subplot_titles=("Tiempo: Delta de Dirac", "Tiempo: Ruido Blanco"))

fig_tiempo.add_trace(go.Scatter(x=t, y=h_delta, mode='lines', name='Delta', line=dict(color='red')), row=1, col=1)
fig_tiempo.add_trace(go.Scatter(x=t, y=h_white, mode='lines', name='Ruido Blanco', line=dict(color='blue')), row=1, col=2)

fig_tiempo.update_layout(
    title_text="Dominio del Tiempo", 
    width=1500, 
    height=500, 
    autosize=False,
    margin=dict(l=50, r=50, t=60, b=50),
    showlegend=False
)
fig_tiempo.show()

* Gráfica la función de autocorrelación $R_{xx}(\tau)$ para la Delta y compárala con la del Ruido Blanco Estocástico. ¿Por qué ambas funciones adoptan una forma equivalente?




In [28]:
fig_autocorr = make_subplots(rows=1, cols=2, subplot_titles=("Autocorrelación: Delta", "Autocorrelación: Ruido Blanco"))

fig_autocorr.add_trace(go.Scatter(x=tau, y=Rxx_delta / np.max(Rxx_delta), mode='lines', line=dict(color='red')), row=1, col=1)
fig_autocorr.add_trace(go.Scatter(x=tau, y=Rxx_white / np.max(Rxx_white), mode='lines', line=dict(color='blue')), row=1, col=2)

fig_autocorr.update_xaxes(range=[-0.1, 0.1])
fig_autocorr.update_layout(
    title_text="Función de Autocorrelación Normalizada", 
    width=1500, 
    height=500, 
    autosize=False,
    margin=dict(l=50, r=50, t=60, b=50),
    showlegend=False
)
fig_autocorr.show()

* Grafica el Espectro de Potencia (PSD) del Pulso Delta y Ruido Blanco Estocástico. Si ambas poseen un espectro plano ($S_{xx}(f) = \text{constante}$), ¿por qué una se manifiesta como un pico extremadamente concentrado y la otra como una señal estocástica e impredecible en el tiempo?  ¿Existe alguna diferencia en cómo se distribuye la energía en el dominio de la frecuencia entre ambas señales?



In [29]:
fig_psd = make_subplots(rows=1, cols=2, subplot_titles=("PSD: Delta de Dirac", "PSD: Ruido Blanco"))

fig_psd.add_trace(go.Scatter(x=f_fft[1:], y=psd_delta[1:], mode='lines', line=dict(color='red')), row=1, col=1)
fig_psd.add_trace(go.Scatter(x=f_fft[1:], y=psd_white[1:], mode='lines', line=dict(color='blue')), row=1, col=2)

fig_psd.update_xaxes(type="log")
fig_psd.update_yaxes(type="log")
fig_psd.update_layout(
    title_text="Densidad Espectral de Potencia (Log-Log)", 
    width=1500, 
    height=500, 
    autosize=False,
    margin=dict(l=50, r=50, t=60, b=50),
    showlegend=False
)
fig_psd.show()

* Grafica el Espectro de Fases del Pulso Delta y Ruido Blanco Estocástico.



In [30]:
fig_fases = make_subplots(rows=1, cols=2, subplot_titles=("Fases: Delta de Dirac", "Fases: Ruido Blanco"))

fig_fases.add_trace(go.Scatter(x=f_fft[1:200], y=fases_delta[1:200], mode='markers', marker=dict(color='red', size=4)), row=1, col=1)
fig_fases.add_trace(go.Scatter(x=f_fft[1:200], y=fases_white[1:200], mode='markers', marker=dict(color='blue', size=4)), row=1, col=2)

fig_fases.update_layout(
    title_text="Espectro de Fases (Limitado a las primeras 200 componentes)", 
    width=1500, 
    height=500, 
    autosize=False,
    margin=dict(l=50, r=50, t=60, b=50),
    showlegend=False
)
fig_fases.show()

* Grafica el comportamiento $H(f)$ del Pulso Delta (rojo) y Ruido Blanco Estocástico (azul) en el plano complejo en coordenadas polares. ¿Qué diferencias se observan?



In [24]:
fig_polar = make_subplots(rows=1, cols=2, specs=[[{'type': 'polar'}, {'type': 'polar'}]],
                          subplot_titles=("Plano Complejo: Delta", "Plano Complejo: Ruido Blanco"))

fig_polar.add_trace(go.Scatterpolar(
    r=np.abs(H_delta), theta=np.angle(H_delta), thetaunit='radians',
    mode='markers', marker=dict(color='red', opacity=0.6, size=5)
), row=1, col=1)

fig_polar.add_trace(go.Scatterpolar(
    r=np.abs(H_white), theta=np.angle(H_white), thetaunit='radians',
    mode='markers', marker=dict(color='blue', opacity=0.6, size=5)
), row=1, col=2)

# La proporción width:height se ajusta a 2:1 para preservar la forma circular perfecta
fig_polar.update_layout(
    title_text="Representación en el Plano Complejo (Magnitud vs Fase)", 
    width=1200, 
    height=600, 
    autosize=False,
    margin=dict(l=50, r=50, t=60, b=50),
    showlegend=False
)
fig_polar.show()

### Memoria en Ruido Rosa ($1/f$)
* Utiliza el sintetizador N-Tori y modifica la atenuación del ruido de color en el código cambiando la amplitud a $A_k = 1 / f_k^\alpha $ (Ruido Rosa y Browniano).

In [33]:
# beta=0 (Blanco), beta=1 (Rosa), beta=2 (Browniano)
t, h_white = base(srate=srate, sec=sec, beta=0, is_dirac=False) 
t, h_pink  = base(srate=srate, sec=sec, beta=1, is_dirac=False) 
t, h_brown = base(srate=srate, sec=sec, beta=2, is_dirac=False)

fig_time = make_subplots(rows=3, cols=1, subplot_titles=("Ruido Blanco (\u03B2=0)", "Ruido Rosa (\u03B2=1)", "Ruido Browniano (\u03B2=2)"))
fig_time.add_trace(go.Scatter(x=t, y=h_white, line=dict(color='blue', width=1), name="Blanco"), row=1, col=1)
fig_time.add_trace(go.Scatter(x=t, y=h_pink, line=dict(color='magenta', width=1), name="Rosa"), row=2, col=1)
fig_time.add_trace(go.Scatter(x=t, y=h_brown, line=dict(color='brown', width=1), name="Browniano"), row=3, col=1)
fig_time.update_layout(height=600, title_text="1. Dominio del Tiempo", showlegend=False)
fig_time.show()

* Grafica los mismos elementos que en el ejercicio pasado para: Ruido Blanco, Ruido Rosa y Ruido Browniano. En el Ruido Blanco, la autocorrelación cae inmediatamente a cero para todo desfase $\tau > 0$. Sin embargo, en el Ruido Rosa o Browniano la función $R_{xx}(\tau)$ decae de forma progresiva. ¿Qué representa esta persistencia en términos de la "memoria temporal" o correlación del sistema?



In [35]:
tau = np.arange(-len(h_white)+1, len(h_white)) / srate
Rxx_white = np.correlate(h_white, h_white, mode='full')
Rxx_pink = np.correlate(h_pink, h_pink, mode='full')
Rxx_brown = np.correlate(h_brown, h_brown, mode='full')

fig_autocorr = make_subplots(rows=1, cols=3, subplot_titles=("Rxx Blanco", "Rxx Rosa", "Rxx Browniano"))
fig_autocorr.add_trace(go.Scatter(x=tau, y=Rxx_white/np.max(Rxx_white), line=dict(color='blue')), row=1, col=1)
fig_autocorr.add_trace(go.Scatter(x=tau, y=Rxx_pink/np.max(Rxx_pink), line=dict(color='magenta')), row=1, col=2)
fig_autocorr.add_trace(go.Scatter(x=tau, y=Rxx_brown/np.max(Rxx_brown), line=dict(color='brown')), row=1, col=3)
fig_autocorr.update_layout(height=400, title_text="2. Función de Autocorrelación Rxx(\u03C4)", showlegend=False)
fig_autocorr.update_xaxes(range=[-0.5, 0.5]) # Zoom inicial en el centro
fig_autocorr.show()


In [38]:
# Espectro de fases 
fases_white = np.angle(np.fft.rfft(h_white))[1:300]
fases_pink = np.angle(np.fft.rfft(h_pink))[1:300]
fases_brown = np.angle(np.fft.rfft(h_brown))[1:300]
f_fases = f_fft[1:300]

fig_fases = make_subplots(rows=1, cols=3, subplot_titles=("Fases Blanco", "Fases Rosa", "Fases Browniano"))
fig_fases.add_trace(go.Scatter(x=f_fases, y=fases_white, mode='markers', marker=dict(size=4, color='blue')), row=1, col=1)
fig_fases.add_trace(go.Scatter(x=f_fases, y=fases_pink, mode='markers', marker=dict(size=4, color='magenta')), row=1, col=2)
fig_fases.add_trace(go.Scatter(x=f_fases, y=fases_brown, mode='markers', marker=dict(size=4, color='brown')), row=1, col=3)
fig_fases.update_layout(height=400, title_text="4. Espectro de Fases", showlegend=False)
fig_fases.show()

In [39]:
# Plano polar 

H_white = np.fft.rfft(h_white)[1:300]
H_pink = np.fft.rfft(h_pink)[1:300]
H_brown = np.fft.rfft(h_brown)[1:300]

fig_polar = make_subplots(rows=1, cols=3, specs=[[{'type': 'polar'}, {'type': 'polar'}, {'type': 'polar'}]],
                          subplot_titles=("Polar Blanco", "Polar Rosa", "Polar Browniano"))
fig_polar.add_trace(go.Scatterpolar(r=np.abs(H_white), theta=np.angle(H_white), mode='markers', marker=dict(size=4, color='blue', opacity=0.6), thetaunit='radians'), row=1, col=1)
fig_polar.add_trace(go.Scatterpolar(r=np.abs(H_pink), theta=np.angle(H_pink), mode='markers', marker=dict(size=4, color='magenta', opacity=0.6), thetaunit='radians'), row=1, col=2)
fig_polar.add_trace(go.Scatterpolar(r=np.abs(H_brown), theta=np.angle(H_brown), mode='markers', marker=dict(size=4, color='brown', opacity=0.6), thetaunit='radians'), row=1, col=3)
fig_polar.update_layout(height=400, title_text="5. Plano Complejo Coordenadas Polares", showlegend=False)
fig_polar.show()

* Estima la pendiente de los espectros de potencia Power vs. f, en escala log-log. ¿Coincide con lo esperado?


In [37]:
f_fft = np.fft.rfftfreq(len(t), d=1/srate)
psd_white = np.abs(np.fft.rfft(h_white))**2
psd_pink = np.abs(np.fft.rfft(h_pink))**2
psd_brown = np.abs(np.fft.rfft(h_brown))**2

mask = f_fft > 0
slope_white, _ = np.polyfit(np.log10(f_fft[mask]), np.log10(psd_white[mask]), 1)
slope_pink, _ = np.polyfit(np.log10(f_fft[mask]), np.log10(psd_pink[mask]), 1)
slope_brown, _ = np.polyfit(np.log10(f_fft[mask]), np.log10(psd_brown[mask]), 1)

print(f"--- ESTIMACIÓN DE PENDIENTES (Power vs f) ---")
print(f"Pendiente Ruido Blanco   : {slope_white:.2f} (Esperado: ~ 0.00)")
print(f"Pendiente Ruido Rosa     : {slope_pink:.2f} (Esperado: ~ -1.00)")
print(f"Pendiente Ruido Browniano: {slope_brown:.2f} (Esperado: ~ -2.00)\n")

fig_psd = make_subplots(rows=1, cols=3, subplot_titles=("PSD Blanco", "PSD Rosa", "PSD Browniano"))
fig_psd.add_trace(go.Scatter(x=f_fft[mask], y=psd_white[mask], mode='lines', line=dict(color='blue')), row=1, col=1)
fig_psd.add_trace(go.Scatter(x=f_fft[mask], y=psd_pink[mask], mode='lines', line=dict(color='magenta')), row=1, col=2)
fig_psd.add_trace(go.Scatter(x=f_fft[mask], y=psd_brown[mask], mode='lines', line=dict(color='brown')), row=1, col=3)
fig_psd.update_xaxes(type="log"); fig_psd.update_yaxes(type="log")
fig_psd.update_layout(height=400, title_text="3. Espectro de Potencias (Escala Log-Log)", showlegend=False)
fig_psd.show()

--- ESTIMACIÓN DE PENDIENTES (Power vs f) ---
Pendiente Ruido Blanco   : 0.37 (Esperado: ~ 0.00)
Pendiente Ruido Rosa     : -0.59 (Esperado: ~ -1.00)
Pendiente Ruido Browniano: -1.56 (Esperado: ~ -2.00)

